In [3]:
# @title Setup Env
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers chromadb \
                langchain langchain-huggingface langchain-chroma langchain-community langchain-ollama \
                --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install pypdf
!pip install langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 28.8 MB/s eta 0:00:00


# 2-stage IR without legacy packages

This is a vibe-coded reimplementation of the original 2-stage IR pipeline without usage of legacy (`langchain_classic`) features, which is (allegedly) meant to be a more modern and interoperable take on the usage of HF rerankers inside LangChain.

While mostly just a proof of concept, this script should be useful as a template or reference for further experiments that may run into similar problems.

In [6]:
import langchain, langchain_core
langchain.__version__, langchain_core.__version__

('1.2.13', '1.2.19')

In [7]:
import langchain, langchain_core
langchain.__version__, langchain_core.__version__

# Made with AdaptaOne
#
# =========================
# Two-stage Information Retrieval
# LangChain 1.2.x + Chroma + Hugging Face
# Stage 1: dense vector retrieval
# Stage 2: cross-encoder reranking
# =========================

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Sequence, Optional, Dict, Any
import os

import torch
from sentence_transformers import CrossEncoder
from pypdf import PdfReader # Added for PDF processing

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
# ============================================================
# Configuration
# ============================================================

@dataclass
class RetrievalConfig:
    # Embedding model for stage 1
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Cross-encoder reranker model for stage 2
    # Good lightweight default for limited consumer GPUs
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Chroma persistence
    persist_directory: str = "./chroma_langchain_db"
    collection_name: str = "basic_ir_collection"

    # Retrieval sizes
    initial_k: int = 20   # retrieved from vector store
    final_k: int = 5      # kept after reranking

    # Device and performance
    device: Optional[str] = None
    normalize_embeddings: bool = True

    # Cross-encoder inference batch size
    # Keep conservative for consumer GPU with limited VRAM
    reranker_batch_size: int = 8


# ============================================================
# Utility functions
# ============================================================

def get_device(explicit_device: Optional[str] = None) -> str:
    if explicit_device:
        return explicit_device
    return "cuda" if torch.cuda.is_available() else "cpu"


def print_ranked_results(
    docs: Sequence[Document],
    score_key: str = "reranker_score",
    max_chars: int = 400,
) -> None:
    if not docs:
        print("No documents found.")
        return

    for idx, doc in enumerate(docs, start=1):
        score = doc.metadata.get(score_key, None)
        source = doc.metadata.get("source", "N/A")

        print("=" * 100)
        print(f"Rank: {idx}")
        print(f"Source: {source}")
        if score is not None:
            print(f"Reranker score: {score:.6f}")
        print("-" * 100)
        print(doc.page_content[:max_chars].strip())
        print()


# ============================================================
# Two-stage retriever
# ============================================================

class TwoStageRetriever:
    """
    Two-stage retrieval pipeline:
    1. Dense retrieval from Chroma using sentence-transformers embeddings
    2. Cross-encoder reranking using sentence-transformers CrossEncoder

    This design keeps LangChain for orchestration and storage integration,
    while using the most stable Hugging Face-native path for reranking.
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size

        self.cross_encoder = CrossEncoder(
            reranker_model_name,
            device=self.device,
        )

    def _dense_retrieve(self, query: str) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        if not docs:
            return []

        pairs = [(query, doc.page_content) for doc in docs]

        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )

        reranked_docs: List[Document] = []
        for doc, score in zip(docs, scores):
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["reranker_score"] = float(score)

            reranked_docs.append(
                Document(
                    page_content=doc.page_content,
                    metadata=metadata,
                )
            )

        reranked_docs.sort(
            key=lambda d: d.metadata["reranker_score"],
            reverse=True,
        )

        return reranked_docs[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        initial_docs = self._dense_retrieve(query)
        final_docs = self._rerank(query, initial_docs)
        return final_docs

    def batch(self, queries: Sequence[str]) -> List[List[Document]]:
        return [self.invoke(query) for query in queries]


# ============================================================
# Build vector store
# ============================================================

def build_embeddings(config: RetrievalConfig) -> HuggingFaceEmbeddings:
    device = get_device(config.device)

    return HuggingFaceEmbeddings(
        model_name=config.embedding_model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
    )


def build_vectorstore(config: RetrievalConfig) -> Chroma:
    embeddings = build_embeddings(config)

    vectorstore = Chroma(
        collection_name=config.collection_name,
        embedding_function=embeddings,
        persist_directory=config.persist_directory,
    )

    return vectorstore


# ============================================================
# Optional ingestion helper
# ============================================================

def add_documents_to_vectorstore(
    vectorstore: Chroma,
    texts: Sequence[str],
    metadatas: Optional[Sequence[Dict[str, Any]]] = None,
) -> None:
    docs = []
    if metadatas is None:
        metadatas = [{} for _ in texts]

    for text, metadata in zip(texts, metadatas):
        docs.append(Document(page_content=text, metadata=metadata))

    vectorstore.add_documents(docs)


# ============================================================
# Optional LCEL-style wrapper
# ============================================================

def build_lcel_rerank_runnable(two_stage_retriever: TwoStageRetriever):
    """
    Optional helper if you want to plug the retriever into a Runnable-based flow later.
    This keeps the implementation interoperable with the LangChain 1.x style.
    """
    from langchain_core.runnables import RunnableLambda

    return RunnableLambda(two_stage_retriever.invoke)


# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    config = RetrievalConfig(
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        reranker_model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        persist_directory="./chroma_langchain_db",
        collection_name="basic_ir_collection",
        initial_k=20,
        final_k=5,
        device=None,               # auto-detects cuda if available
        normalize_embeddings=True,
        reranker_batch_size=8,
    )

    print(f"Using device: {get_device(config.device)}")

    vectorstore = build_vectorstore(config)

    # --------------------------------------------------------
    # OPTIONAL: only run this once if your collection is empty
    # --------------------------------------------------------

    # --- PDF Loading Logic ---
    pdf_file_path = '/content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf'
    texts_from_pdf = []
    metadatas_from_pdf = []
    pdf_loaded_successfully = False

    try:
        reader = PdfReader(pdf_file_path)
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,  # Max characters per chunk
            chunk_overlap=200, # Overlap for context between chunks
            separators=["\n\n", "\n", ".", "!", "?", ",", " "] # Attempt to split by sentences/paragraphs first
        )

        for i, page in enumerate(reader.pages):
            page_content = page.extract_text()
            if page_content: # Only add if page has content
                # Split page content into chunks
                chunks = text_splitter.split_text(page_content)
                for chunk_idx, chunk in enumerate(chunks):
                    texts_from_pdf.append(chunk)
                    # Assign metadata to each chunk, indicating its original page and chunk number
                    metadatas_from_pdf.append({"source": f"{pdf_file_path}_page_{i+1}_chunk_{chunk_idx+1}"})
        if texts_from_pdf:
            print(f"Successfully loaded and chunked {len(texts_from_pdf)} chunks from PDF: {pdf_file_path}")
            pdf_loaded_successfully = True
        else:
            print(f"PDF loaded but no text extracted or chunks created from {pdf_file_path}. Falling back to sample texts.")

    except FileNotFoundError:
        print(f"PDF file not found at '{pdf_file_path}'. Falling back to sample texts.")
    except Exception as e:
        print(f"An error occurred while loading PDF: {e}. Falling back to sample texts.")


    if pdf_loaded_successfully:
        texts_to_add = texts_from_pdf
        metadatas_to_add = metadatas_from_pdf
    else:
        sample_texts = [
    # "O que diz a cláusula 18 sobre a remuneração de horas extras?"
    # "Os comerciários com salários superiores à R$ 11.000,00 (onze mil reais), mediante livre negociação"
    # "As horas extras diárias serão remuneradas com o adicional legal de 60% (sessenta por cento), incidindo"
    # "Se as férias foram concedidas entre 1º/09/2025 e a assinatura da convenção, as diferenças salariais"

        ]

        sample_metadatas = [
            {"source": "doc_1"},
            {"source": "doc_2"},
            {"source": "doc_3"},
            {"source": "doc_4"},
            {"source": "doc_5"},
        ]
        texts_to_add = sample_texts
        metadatas_to_add = sample_metadatas


    add_documents_to_vectorstore(vectorstore, texts_to_add, metadatas_to_add)

    retriever = TwoStageRetriever(
        vectorstore=vectorstore,
        reranker_model_name=config.reranker_model_name,
        initial_k=config.initial_k,
        final_k=config.final_k,
        device=config.device,
        reranker_batch_size=config.reranker_batch_size,
    )

    query = "CARGO 6: PERITO CRIMINAL FEDERAL – ÁREA 7: ENGENHARIA CIVIL?"
    results = retriever.invoke(query)

    print_ranked_results(results)

    # Optional LCEL-compatible runnable
    # runnable_retriever = build_lcel_rerank_runnable(retriever)
    # runnable_results = runnable_retriever.invoke(query)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Rank: 1
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_41_chunk_4
Reranker score: 9.195508
----------------------------------------------------------------------------------------------------
Forense 17 6 6 
Cargo 6: Perito Criminal Federal – Área 7: 
Engenharia Civil 11 6 6 
Cargo 7: Perito Criminal Federal – Área 11: 
Engenharia Cartográfica 6 6 6 
Cargo 8: Perito Criminal Federal – Área 12: 
Medicina Legal 6 6 6 
Cargo 9: Perito Criminal Federal – Área 16: Física 
Forense 6 6 6

Rank: 2
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_29_chunk_4
Reranker score: 8.835571
----------------------------------------------------------------------------------------------------
Elétrica/Eletrônica 6 6 6 
Cargo 4: Perito Criminal Federal – Área 3: Informática Forense 69 11 69 
Cargo 5: Perito Criminal Federal – Área 5: Geologia Forense 17 6 17 
Cargo 6: Perito Criminal Federal – Área 7: Engenharia Civil 11 6 11 
Cargo 7: Perito Criminal Federal – Área

In [8]:
new_query = "CARGO 7: PERITO CRIMINAL FEDERAL – ÁREA 11: ENGENHARIA CARTOGRÁFICA"
new_results = retriever.invoke(new_query)

print(f"\nResults for query: '{new_query}'")
print_ranked_results(new_results)


Results for query: 'CARGO 7: PERITO CRIMINAL FEDERAL – ÁREA 11: ENGENHARIA CARTOGRÁFICA'
Rank: 1
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_41_chunk_4
Reranker score: 9.150510
----------------------------------------------------------------------------------------------------
Forense 17 6 6 
Cargo 6: Perito Criminal Federal – Área 7: 
Engenharia Civil 11 6 6 
Cargo 7: Perito Criminal Federal – Área 11: 
Engenharia Cartográfica 6 6 6 
Cargo 8: Perito Criminal Federal – Área 12: 
Medicina Legal 6 6 6 
Cargo 9: Perito Criminal Federal – Área 16: Física 
Forense 6 6 6

Rank: 2
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_9_chunk_2
Reranker score: 9.128271
----------------------------------------------------------------------------------------------------
Cargo 5: Perito Criminal Federal 
– Área 5: Geologia Forense 3 1 1 5 
Cargo 6: Perito Criminal Federal 
– Área 7: Engenharia Civil 2 * * 2 
Cargo 7: Perito Criminal Federal 
– Área 11: Eng

In [9]:
new_query = "CARGO 9: PERITO CRIMINAL FEDERAL – ÁREA 16: FÍSICA FORENSE"
new_results = retriever.invoke(new_query)

print(f"\nResults for query: '{new_query}'")
print_ranked_results(new_results)


Results for query: 'CARGO 9: PERITO CRIMINAL FEDERAL – ÁREA 16: FÍSICA FORENSE'
Rank: 1
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_47_chunk_2
Reranker score: 9.275080
----------------------------------------------------------------------------------------------------
Cargo 8: Perito Criminal Federal – Área 12: 
Medicina Legal 6 6 6 
Cargo 9: Perito Criminal Federal – Área 16: Física 
Forense 6 6 6 
Cargo 10: Perito Criminal Federal – Área 17: 
Engenharia de Minas 6 6 6 
Cargo 11: Perito Criminal Federal – Área 19: 
Genética Forense 6 6 6 
Cargo 12: Perito Criminal Federal – Área 20: 
Engenharia Ambiental 6 6 6 
Cargo 13: Perito Criminal Federal – Área 21: 
Ant

Rank: 2
Source: /content/drive/MyDrive/UC15/PDF/Ed_1_PF_25_Abertura.pdf_page_5_chunk_2
Reranker score: 9.036112
----------------------------------------------------------------------------------------------------
CARGO 9: PERITO CRIMINAL FEDERAL – ÁREA 16: FÍSICA FORENSE 
REQUISITO: diploma, devidament